<a href="https://colab.research.google.com/github/Anshgupta2906/AI_agent_from_scratch/blob/main/02_tool_pattern/TOOL_pattern_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install openai requests -q

In [28]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

In [29]:
def convert_units(value,convert_to):
  """convert celcius to Fahrenheit, miles to kilometer, or kilometers to miles"""
  if convert_to=="Fahrenheit":
    return (value*9/5)+32
  elif convert_to=="kilometer":
    return value*1.609
  elif convert_to=="mile":
    return value*0.621371 # 1 kilometer = 0.621371 miles
  else:
    return "unknown conversion"

print(convert_units(3,"kilometer"))

4.827


In [30]:
tools=[
    {
        "type":"function",
        "function":{
            "name":"convert_units",
            "description":"convert celcius to Fahrenheit, miles to kilometer, or kilometers to miles",
            "parameters":{
                "type":"object",
                "properties":{
                    "value":{
                        "type":"number",
                        "description":"The numerical value to convert"
                    },
                    "convert_to":{
                        "type":"string",
                        "enum": ["Fahrenheit", "kilometer", "mile"],
                        "description":"The unit to convert to (e.g., Fahrenheit, kilometer, or mile)"
                    }
                },
                "required":["value","convert_to"]
            }
        }
    }
]

In [31]:
Response=client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{
        "role":"user","content":"convert 10 kilometers to miles"}
    ],
    tools=tools,
    tool_choice="auto"

)
print(Response)

ChatCompletion(id='gen-1775293943-u35FUCzpiMKKq3wD8d2R', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-f533fd77de5b48bda9c0a711efb993bf', function=Function(arguments='{"value": 10, "convert_to": "mile"}', name='convert_units'), type='function', index=0)], reasoning='Okay, the user wants to convert 10 kilometers to miles. Let me check the available tools. There\'s a convert_units function that handles kilometers to miles. The parameters needed are value and convert_to. The value here is 10, and convert_to should be "mile". I need to make sure the function is called correctly with these parameters. No other tools are available, so this should be straightforward.\n', reasoning_details=[{'type': 'reasoning.text', 'text': 'Okay, the user wants to convert 10 kilometers to miles. 

In [32]:
import json

# Get the message from the response
message = Response.choices[0].message

# Check if the model actually provided tool_calls
if message.tool_calls:
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    print(f"Function to call: {function_name}")
    print(f"Arguments: {arguments}")

    # Execute the function
    result = convert_units(**arguments)
    print(f"Tool execution result: {result}")
else:
    print("The model did not return a formal tool call. It returned text instead:")
    print(message.content)

Function to call: convert_units
Arguments: {'value': 10, 'convert_to': 'mile'}
Tool execution result: 6.21371


In [33]:
if 'tool_call' in globals():
    messages=[
        {"role":"user","content":"convert 10 kilometers to miles"},
        Response.choices[0].message,
        {
            "role":"tool",
            "tool_call_id":tool_call.id,
            "content":str(result)
        }
    ]
    Final_response=client.chat.completions.create(
        model=FREE_MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    print(Final_response.choices[0].message.content)
else:
    print("Cannot proceed: No tool call was captured in the previous step.")

10 kilometers is equal to approximately 6.21 miles.


In [ ]:
def run_conversation(user_prompt):
    # 1. Setup the conversation
    messages = [{"role": "user", "content": user_prompt}]

    # 2. First LLM Call
    response = client.chat.completions.create(
        model=FREE_MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    response_message = response.choices[0].message

    # 3. Check for tool calls
    if response_message.tool_calls:
        messages.append(response_message)

        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            # Map function name string to actual Python function
            if function_name == "convert_units":
                function_response = convert_units(**function_args)

                # 4. Add result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": str(function_response)
                })

        # 5. Final LLM Call with results
        final_response = client.chat.completions.create(
            model=FREE_MODEL,
            messages=messages
        )
        return final_response.choices[0].message.content

    return response_message.content

# Test the automated loop
print(run_conversation("How many miles is 50 kilometers?"))